# 2D gastruloid analysis — chemistry-aware joint SCVI workflow

This notebook preserves the Amanda/Liang loading and QC workflow, then jointly trains the published gastruloid reference and query cells to account for library-chemistry and depth differences.


In [ ]:
from pathlib import Path
import re
import sys
import types
import warnings

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.neighbors import KNeighborsClassifier

import scvi

print("Python:", sys.version.split()[0])
print("scanpy:", sc.__version__)
print("anndata:", ad.__version__)
print("scvi-tools:", scvi.__version__)

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)


## 1. Paths and editable settings

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd().parent / "utils"))   # run notebooks from their own folder
from config import PROJECT_DIR, CELLRANGER_DIR

CELLRANGER_BASE = CELLRANGER_DIR
ANALYSIS_DIR = PROJECT_DIR / "new_analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

RAW_COMBINED_PATH = ANALYSIS_DIR / "amanda_liang_raw_unfiltered.h5ad"

# Adjust these two paths only if your downloaded reference/model are elsewhere.
REFERENCE_CANDIDATES = [
    (PROJECT_DIR / "adata_timeseries_all_cc_hvg1000_leiden10_23_v3.h5ad"),
    Path.home() / "human-gastruloid-analysis" / "adata_timeseries_all_cc_hvg1000_leiden10_23_v3.h5ad",
    Path.home() / "human-gastruloid-analysis" / "data" / "adata_timeseries_all_cc_hvg1000_leiden10_23_v3.h5ad",
]

MODEL_CANDIDATES = [
    Path.home() / "human-gastruloid-analysis" / "240306_alltimeseries_cc_scvi_model_487296_hvg1000",
    Path.home() / "human-gastruloid-analysis" / "models" / "240306_alltimeseries_cc_scvi_model_487296_hvg1000",
    Path.home() / "human-gastruloid-analysis" / "src" / "human_gastruloid_analysis" / "models" / "240306_alltimeseries_cc_scvi_model_487296_hvg1000",
    (PROJECT_DIR / "240306_alltimeseries_cc_scvi_model_487296_hvg1000"),
]

def first_existing(candidates, label):
    for p in candidates:
        if p.exists():
            print(f"{label}: {p}")
            return p
    raise FileNotFoundError(
        f"Could not locate {label}. Checked:\n" +
        "\n".join(str(p) for p in candidates)
    )

REFERENCE_H5AD = first_existing(REFERENCE_CANDIDATES, "Reference h5ad")
MODEL_DIR = first_existing(MODEL_CANDIDATES, "SCVI model")

# Conservative starting QC values.
MIN_GENES = 500
MIN_COUNTS = 1_000
MAX_PCT_MT = 20.0       # matches the original notebook
UPPER_MAD = 5.0         # per-sample high-count/high-gene outlier threshold

QUERY_MAX_EPOCHS = 100
KNN_K = 25


## 2. Load the eight per-sample Cell Ranger matrices

In [ ]:
sample_table = pd.DataFrame([
    ("Chicago_Ctrl_48h", "Chicago", "Ctrl", "48h",
     CELLRANGER_BASE / "Chicago_ocm_outs/per_sample_outs/Ctrl_48h_OB1/sample_filtered_feature_bc_matrix.h5"),
    ("Chicago_Ctrl_72h", "Chicago", "Ctrl", "72h",
     CELLRANGER_BASE / "Chicago_ocm_outs/per_sample_outs/Ctrl_72h_OB2/sample_filtered_feature_bc_matrix.h5"),
    ("Chicago_DOX_48h", "Chicago", "DOX", "48h",
     CELLRANGER_BASE / "Chicago_ocm_outs/per_sample_outs/DOX_48h_OB3/sample_filtered_feature_bc_matrix.h5"),
    ("Chicago_DOX_72h", "Chicago", "DOX", "72h",
     CELLRANGER_BASE / "Chicago_ocm_outs/per_sample_outs/DOX_72h_OB4/sample_filtered_feature_bc_matrix.h5"),
    ("Michigan_Ctrl_48h", "Michigan", "Ctrl", "48h",
     CELLRANGER_BASE / "Michigan_ocm_outs/per_sample_outs/Ctrl_48h_OB1/sample_filtered_feature_bc_matrix.h5"),
    ("Michigan_Ctrl_72h", "Michigan", "Ctrl", "72h",
     CELLRANGER_BASE / "Michigan_ocm_outs/per_sample_outs/Ctrl_72h_OB2/sample_filtered_feature_bc_matrix.h5"),
    ("Michigan_DOX_48h", "Michigan", "DOX", "48h",
     CELLRANGER_BASE / "Michigan_ocm_outs/per_sample_outs/DOX_48h_OB3/sample_filtered_feature_bc_matrix.h5"),
    ("Michigan_DOX_72h", "Michigan", "DOX", "72h",
     CELLRANGER_BASE / "Michigan_ocm_outs/per_sample_outs/DOX_72h_OB4/sample_filtered_feature_bc_matrix.h5"),
], columns=["sample_id", "site", "condition", "time", "path"])

missing = sample_table.loc[~sample_table["path"].map(Path.exists)]
if len(missing):
    raise FileNotFoundError(str(missing[["sample_id", "path"]]))

display(sample_table)


In [ ]:
def load_one_sample(row):
    a = sc.read_10x_h5(row.path)

    if "gene_ids" not in a.var.columns:
        a.var["gene_ids"] = a.var_names.astype(str)

    a.var_names_make_unique()

    a.obs["sample_id"] = row.sample_id
    a.obs["sample_labels"] = row.sample_id  # compatible with original notebook naming
    a.obs["site"] = row.site
    a.obs["condition"] = row.condition
    a.obs["time"] = row.time
    a.obs["condition_time"] = f"{row.condition}_{row.time}"

    a.obs_names = row.sample_id + "_" + a.obs_names.astype(str)
    return a

if RAW_COMBINED_PATH.exists():
    print("Using previously created raw object:")
    print(RAW_COMBINED_PATH)
    adata_raw = sc.read_h5ad(RAW_COMBINED_PATH)
else:
    pieces = [load_one_sample(r) for r in sample_table.itertuples(index=False)]
    adata_raw = ad.concat(pieces, join="outer", merge="first", index_unique=None)
    adata_raw.layers["counts"] = adata_raw.X.copy()
    adata_raw.write_h5ad(RAW_COMBINED_PATH)
    print("Saved:", RAW_COMBINED_PATH)

if "counts" not in adata_raw.layers:
    adata_raw.layers["counts"] = adata_raw.X.copy()

print(adata_raw)
display(adata_raw.obs["sample_id"].value_counts().sort_index())


## 3. QC metrics and filtering

In [ ]:
adata_raw.var["mt"] = adata_raw.var_names.str.upper().str.startswith("MT-")
adata_raw.var["ribo"] = adata_raw.var_names.str.upper().str.startswith(("RPS", "RPL"))
adata_raw.var["hb"] = adata_raw.var_names.str.upper().str.match(r"^HB(?!P)")

needed = {"n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo"}
if not needed.issubset(adata_raw.obs.columns):
    sc.pp.calculate_qc_metrics(
        adata_raw,
        qc_vars=["mt", "ribo", "hb"],
        percent_top=[20, 50, 100],
        log1p=True,
        inplace=True,
    )

qc_summary = (
    adata_raw.obs.groupby("sample_id", observed=True)
    .agg(
        n_cells=("sample_id", "size"),
        median_genes=("n_genes_by_counts", "median"),
        median_counts=("total_counts", "median"),
        median_pct_mt=("pct_counts_mt", "median"),
        median_pct_ribo=("pct_counts_ribo", "median"),
        p01_counts=("total_counts", lambda x: x.quantile(.01)),
        p99_counts=("total_counts", lambda x: x.quantile(.99)),
    )
)
display(qc_summary)
qc_summary.to_csv(ANALYSIS_DIR / "qc_summary_before_filtering.csv")


In [ ]:
for metric in ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo"]:
    sc.pl.violin(
        adata_raw,
        keys=metric,
        groupby="sample_id",
        rotation=90,
        stripplot=False,
    )


In [ ]:
def mad(x):
    x = np.asarray(x, dtype=float)
    med = np.nanmedian(x)
    return np.nanmedian(np.abs(x - med))

obs = adata_raw.obs.copy()
obs["fail_low_genes"] = obs["n_genes_by_counts"] < MIN_GENES
obs["fail_low_counts"] = obs["total_counts"] < MIN_COUNTS
obs["fail_mt"] = obs["pct_counts_mt"] > MAX_PCT_MT
obs["fail_upper_genes"] = False
obs["fail_upper_counts"] = False

threshold_rows = []

for sample_id, idx in obs.groupby("sample_id", observed=True).groups.items():
    g = obs.loc[idx]

    g_med = g["n_genes_by_counts"].median()
    g_mad = mad(g["n_genes_by_counts"])
    c_med = g["total_counts"].median()
    c_mad = mad(g["total_counts"])

    g_max = g_med + UPPER_MAD * g_mad
    c_max = c_med + UPPER_MAD * c_mad

    if g_mad > 0:
        obs.loc[idx, "fail_upper_genes"] = g["n_genes_by_counts"] > g_max
    if c_mad > 0:
        obs.loc[idx, "fail_upper_counts"] = g["total_counts"] > c_max

    threshold_rows.append({
        "sample_id": sample_id,
        "max_genes": g_max,
        "max_counts": c_max,
    })

obs["pass_qc"] = ~obs[
    ["fail_low_genes", "fail_low_counts", "fail_mt",
     "fail_upper_genes", "fail_upper_counts"]
].any(axis=1)

for col in obs.columns:
    if col not in adata_raw.obs.columns:
        adata_raw.obs[col] = obs[col]

filter_report = (
    adata_raw.obs.groupby("sample_id", observed=True)
    .agg(
        before=("sample_id", "size"),
        retained=("pass_qc", "sum"),
        low_genes=("fail_low_genes", "sum"),
        low_counts=("fail_low_counts", "sum"),
        high_mt=("fail_mt", "sum"),
        high_gene_outlier=("fail_upper_genes", "sum"),
        high_count_outlier=("fail_upper_counts", "sum"),
    )
)
filter_report["removed"] = filter_report["before"] - filter_report["retained"]
filter_report["pct_retained"] = 100 * filter_report["retained"] / filter_report["before"]

display(filter_report)
display(pd.DataFrame(threshold_rows).set_index("sample_id"))
filter_report.to_csv(ANALYSIS_DIR / "qc_filter_report.csv")


The original notebook used a fixed library-size window of 8,000–25,000 counts. That was appropriate for its historical samples but is not directly portable: Chicago Ctrl 72 h and Michigan DOX 72 h have median counts below 8,000. The modified rule retains plausible lower-depth cells and removes only clear low-information cells plus sample-specific extreme upper tails.

In [ ]:
adata_full = adata_raw[adata_raw.obs["pass_qc"].to_numpy()].copy()
adata_full.X = adata_full.layers["counts"].copy()
adata_full.layers["counts"] = adata_full.X.copy()

FULL_QC_PATH = ANALYSIS_DIR / "amanda_liang_fullgenes_qc_filtered.h5ad"
adata_full.write_h5ad(FULL_QC_PATH)

print(adata_full)
print("Saved:", FULL_QC_PATH)


## 4. Exploratory normalization, HVGs, and cell-cycle nuisance scoring

The original notebook selected 1,000 HVGs, then excluded cell-cycle genes from the SCVI feature matrix and included their expression as continuous covariates. It did not contain a ribosomal-gene exclusion step. Here, an exploratory object is created without changing the full count object. Technical gene families are excluded from *de novo* HVG selection only.

In [ ]:
adata_explore = adata_full.copy()
adata_explore.X = adata_explore.layers["counts"].copy()

sc.pp.normalize_total(adata_explore, target_sum=1e4)
sc.pp.log1p(adata_explore)

sc.pp.highly_variable_genes(
    adata_explore,
    n_top_genes=1_000,
    batch_key="sample_id",
    flavor="seurat_v3",
    layer="counts",
)

technical = (
    adata_explore.var["mt"].fillna(False)
    | adata_explore.var["ribo"].fillna(False)
    | adata_explore.var["hb"].fillna(False)
)
adata_explore.var.loc[technical, "highly_variable"] = False

print("Nontechnical HVGs:", int(adata_explore.var["highly_variable"].sum()))

sc.tl.score_genes_cell_cycle(
    adata_explore,
    s_genes=[g for g in sc.queries.mitochondrial_genes("hsapiens")[:0]],  # placeholder avoided
    g2m_genes=[],
) if False else None

# Standard Scanpy cell-cycle gene sets can be supplied here later if exact
# Macosko_cell_cycle_genes.txt is recovered. Mapping below does not require it.

sc.pp.scale(adata_explore, max_value=10)
sc.tl.pca(adata_explore, mask_var="highly_variable")
sc.pp.neighbors(adata_explore, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata_explore)

sc.pl.umap(
    adata_explore,
    color=["sample_id", "site", "condition", "time",
           "pct_counts_mt", "pct_counts_ribo"],
    wspace=0.4,
)


## 5. Load the validated gastruloid reference and SCVI model

In [ ]:
reference = sc.read_h5ad(REFERENCE_H5AD)

print(reference)
print("Reference obs columns:", reference.obs.columns.tolist())
print("Reference obsm:", list(reference.obsm.keys()))
print("Reference layers:", list(reference.layers.keys()))
display(reference.obs["leiden_sub"].value_counts())

# Compatibility shim required by the older serialized model under modern pandas.
numeric_module = types.ModuleType("pandas.core.indexes.numeric")
numeric_module.Int64Index = pd.Index
numeric_module.UInt64Index = pd.Index
numeric_module.Float64Index = pd.Index
sys.modules["pandas.core.indexes.numeric"] = numeric_module

with warnings.catch_warnings():
    warnings.simplefilter("default")
    ref_model = scvi.model.SCVI.load(MODEL_DIR, adata=reference)

model_genes = pd.Index(ref_model.get_var_names())
reference_genes = pd.Index(reference.var_names)

print("Model genes:", len(model_genes))
print("Reference genes:", len(reference_genes))
print("Exact same order:", model_genes.equals(reference_genes))

if not model_genes.equals(reference_genes):
    raise ValueError("Reference AnnData and SCVI model genes do not match exactly.")


## 6. Construct the exact 963-gene query object

In [ ]:
def parse_ref_name(name):
    name = str(name)
    m = re.match(r"^(.*?)\s*\((ENSG\d+)(?:\.\d+)?\)$", name)
    if m:
        return m.group(1).strip(), m.group(2)
    return name.strip(), None

query_symbols = pd.Index(adata_full.var_names.astype(str))

if "gene_ids" in adata_full.var.columns:
    query_ensembl = pd.Index(
        adata_full.var["gene_ids"].astype(str).str.replace(r"\.\d+$", "", regex=True)
    )
else:
    query_ensembl = query_symbols

symbol_to_pos = {}
ensembl_to_pos = {}

for i, symbol in enumerate(query_symbols):
    symbol_to_pos.setdefault(symbol, i)
for i, ens in enumerate(query_ensembl):
    ensembl_to_pos.setdefault(ens, i)

positions = []
match_type = []
missing = []

for ref_gene in model_genes:
    symbol, ens = parse_ref_name(ref_gene)

    if ens is not None and ens in ensembl_to_pos:
        positions.append(ensembl_to_pos[ens])
        match_type.append("Ensembl")
    elif symbol in symbol_to_pos:
        positions.append(symbol_to_pos[symbol])
        match_type.append("symbol")
    else:
        positions.append(None)
        match_type.append("missing")
        missing.append(ref_gene)

display(pd.Series(match_type).value_counts())
print("Missing genes:", len(missing))
print(missing[:30])

if missing:
    raise ValueError("Reference genes are missing from the new Cell Ranger matrices.")

X_query = adata_full.layers["counts"][:, positions].copy()

query = ad.AnnData(
    X=X_query,
    obs=adata_full.obs.copy(),
    var=pd.DataFrame(index=model_genes.copy()),
)
query.layers["counts"] = query.X.copy()

assert pd.Index(query.var_names).equals(model_genes)

QUERY_COUNTS_PATH = ANALYSIS_DIR / "amanda_liang_query_963genes_counts.h5ad"
query.write_h5ad(QUERY_COUNTS_PATH)

print(query)
print("Saved:", QUERY_COUNTS_PATH)


## 7. Chemistry-aware joint SCVI integration

This section replaces the frozen scArches query mapping. It trains a **new SCVI model from scratch** on the published gastruloid reference plus Amanda/Liang cells in one shared latent space.

Each biological library is treated as its own batch. This allows the model to absorb reference-versus-query chemistry, depth, site, and run differences without deleting any cells. The published `leiden_sub` labels are used **only after training** for nearest-neighbor label transfer.

The original notebook and model are not modified. All outputs from this notebook include `joint_scvi` in the filename.


In [ ]:
# Confirm that true count matrices are available for both datasets.
# SCVI must be trained on raw integer-like counts, not log-normalized expression.

def choose_count_matrix(adata, name):
    if "counts" in adata.layers:
        matrix = adata.layers["counts"]
        source = "layers['counts']"
    elif getattr(adata, "raw", None) is not None and adata.raw.shape[1] == adata.shape[1]:
        matrix = adata.raw.X
        source = "raw.X"
    else:
        matrix = adata.X
        source = "X"

    # Light diagnostic rather than converting the entire sparse matrix.
    sample = matrix[: min(1000, matrix.shape[0]), : min(200, matrix.shape[1])]
    if sparse.issparse(sample):
        values = sample.data
    else:
        values = np.asarray(sample).ravel()
    values = values[np.isfinite(values)]

    non_integer_fraction = (
        np.mean(np.abs(values - np.round(values)) > 1e-6)
        if values.size else 0.0
    )
    print(f"{name}: using {source}; non-integer fraction in diagnostic sample = {non_integer_fraction:.4f}")

    if non_integer_fraction > 0.01:
        raise ValueError(
            f"{name} does not appear to contain raw counts in {source}. "
            "Do not continue with joint SCVI until the reference count matrix is located."
        )
    return matrix

X_ref_counts = choose_count_matrix(reference, "Reference")
X_query_counts = choose_count_matrix(query, "Query")


In [ ]:
# Build reference and query objects with exactly the same 963 genes and raw counts.
reference_joint = ad.AnnData(
    X=X_ref_counts.copy(),
    obs=reference.obs.copy(),
    var=pd.DataFrame(index=model_genes.copy()),
)
reference_joint.layers["counts"] = reference_joint.X.copy()
reference_joint.obs["dataset"] = "Reference"
reference_joint.obs["known_leiden_sub"] = reference_joint.obs["leiden_sub"].astype(str)

# Keep every reference library separate as a batch.
reference_joint.obs["joint_batch"] = (
    "Reference__" + reference_joint.obs["sample_labels"].astype(str)
)

query_joint = ad.AnnData(
    X=query.layers["counts"].copy(),
    obs=query.obs.copy(),
    var=pd.DataFrame(index=model_genes.copy()),
)
query_joint.layers["counts"] = query_joint.X.copy()
query_joint.obs["dataset"] = "Amanda_Liang"
query_joint.obs["known_leiden_sub"] = "UNLABELED_QUERY"

# sample_id already distinguishes Chicago/Michigan, condition, and time.
query_joint.obs["joint_batch"] = (
    "Query__" + query_joint.obs["sample_id"].astype(str)
)

# Prefix names to guarantee uniqueness when concatenating.
reference_joint.obs_names = "REF__" + reference_joint.obs_names.astype(str)
query_joint.obs_names = "QUERY__" + query_joint.obs_names.astype(str)

joint = ad.concat(
    [reference_joint, query_joint],
    axis=0,
    join="inner",
    merge="same",
    label="joint_source",
    keys=["reference", "query"],
    index_unique=None,
)
joint.layers["counts"] = joint.X.copy()
joint.obs["joint_batch"] = joint.obs["joint_batch"].astype("category")

print(joint)
print("Reference cells:", int((joint.obs["dataset"] == "Reference").sum()))
print("Query cells:", int((joint.obs["dataset"] == "Amanda_Liang").sum()))
print("Batches:")
display(joint.obs["joint_batch"].value_counts())


In [ ]:
# Train a fresh model. Sample-level batches explicitly model technical differences.
JOINT_MODEL_DIR = ANALYSIS_DIR / "reference_amanda_liang_joint_scvi_model"
JOINT_CHECKPOINT = ANALYSIS_DIR / "reference_amanda_liang_joint_963genes.h5ad"

scvi.model.SCVI.setup_anndata(
    joint,
    layer="counts",
    batch_key="joint_batch",
)

if (JOINT_MODEL_DIR / "model.pt").exists():
    print("Loading existing joint model; training will not be repeated.")
    joint_model = scvi.model.SCVI.load(JOINT_MODEL_DIR, adata=joint)
else:
    print("Training a new joint SCVI model.")
    joint_model = scvi.model.SCVI(
        joint,
        n_latent=30,
        n_layers=2,
        n_hidden=128,
        gene_likelihood="nb",
    )
    joint_model.train(
        max_epochs=250,
        early_stopping=True,
        check_val_every_n_epoch=10,
    )

joint.obsm["X_scVI_joint"] = joint_model.get_latent_representation()
joint_model.save(JOINT_MODEL_DIR, overwrite=True, save_anndata=True)
joint.write_h5ad(JOINT_CHECKPOINT)

print("Joint latent shape:", joint.obsm["X_scVI_joint"].shape)
print("Saved model:", JOINT_MODEL_DIR)
print("Saved checkpoint:", JOINT_CHECKPOINT)


## 8. Transfer reference labels in the joint latent space

The classifier is trained only on published reference cells. Query cells receive labels from nearby reference cells in the newly trained chemistry-aware latent space.


In [ ]:
ref_mask = joint.obs["dataset"].eq("Reference").to_numpy()
query_mask = joint.obs["dataset"].eq("Amanda_Liang").to_numpy()

X_ref_joint = np.asarray(joint.obsm["X_scVI_joint"][ref_mask], dtype=np.float32)
X_query_joint = np.asarray(joint.obsm["X_scVI_joint"][query_mask], dtype=np.float32)
y_ref = joint.obs.loc[ref_mask, "known_leiden_sub"].astype(str).to_numpy()

knn_joint = KNeighborsClassifier(
    n_neighbors=KNN_K,
    weights="distance",
    metric="euclidean",
    algorithm="brute",
    n_jobs=-1,
)
knn_joint.fit(X_ref_joint, y_ref)

joint_predictions = knn_joint.predict(X_query_joint)
joint_probabilities = knn_joint.predict_proba(X_query_joint)
joint_confidence = joint_probabilities.max(axis=1)

query_obs_names_prefixed = joint.obs_names[query_mask]
query_original_names = pd.Index([x.removeprefix("QUERY__") for x in query_obs_names_prefixed])

query_joint_result = query.copy()
if not query_joint_result.obs_names.equals(query_original_names):
    query_joint_result = query_joint_result[query_original_names].copy()

query_joint_result.obs["predicted_leiden_sub_joint"] = joint_predictions
query_joint_result.obs["prediction_confidence_joint"] = joint_confidence
query_joint_result.obsm["X_scVI_joint"] = X_query_joint

# Save all class probabilities for transparent diagnostics.
for i, label in enumerate(knn_joint.classes_):
    safe_label = str(label).replace("/", "_").replace(" ", "_")
    query_joint_result.obs[f"prob_joint_{safe_label}"] = joint_probabilities[:, i]

counts_joint = pd.crosstab(
    query_joint_result.obs["sample_id"],
    query_joint_result.obs["predicted_leiden_sub_joint"],
)
fractions_joint = pd.crosstab(
    query_joint_result.obs["sample_id"],
    query_joint_result.obs["predicted_leiden_sub_joint"],
    normalize="index",
)

display(counts_joint)
display(fractions_joint.round(3))
counts_joint.to_csv(ANALYSIS_DIR / "joint_scvi_cellstate_counts_by_sample.csv")
fractions_joint.to_csv(ANALYSIS_DIR / "joint_scvi_cellstate_fractions_by_sample.csv")


## 9. UMAP and chemistry-mixing diagnostics


In [ ]:
sc.pp.neighbors(joint, use_rep="X_scVI_joint", n_neighbors=20)
sc.tl.umap(joint, min_dist=0.3)

# Add the transferred labels only to query rows for plotting.
joint.obs["plot_label"] = joint.obs["known_leiden_sub"].astype(str)
joint.obs.loc[query_mask, "plot_label"] = joint_predictions
joint.obs.loc[query_mask, "prediction_confidence_joint"] = joint_confidence

sc.pl.umap(
    joint,
    color=["dataset", "joint_batch", "plot_label"],
    wspace=0.4,
)

q_start = int(ref_mask.sum())
query_joint_result.obsm["X_umap_joint"] = joint.obsm["X_umap"][query_mask].copy()

sc.pl.embedding(
    query_joint_result,
    basis="umap_joint",
    color=[
        "predicted_leiden_sub_joint",
        "sample_id",
        "site",
        "condition",
        "time",
        "prediction_confidence_joint",
    ],
    wspace=0.4,
)


In [ ]:
sc.pl.umap(
    combined_joint,
    color=["dataset", "known_leiden_sub", "predicted_leiden_sub_joint"],
    wspace=0.4,
)

## 10. Critical validation: did AE-LC and DE return?

This table is the primary decision point. A chemistry-aware model should recover plausible AE-LC/DE populations if the earlier failure was caused mainly by integration mismatch. Marker validation is still required before publication.


In [ ]:
labels_to_check = ["AE-LC", "DE", "PGC-LC", "CS-LC"]
summary_rows = []
for label in labels_to_check:
    prob_col = f"prob_joint_{label}"
    assigned = query_joint_result.obs["predicted_leiden_sub_joint"].eq(label)
    summary_rows.append({
        "label": label,
        "n_assigned": int(assigned.sum()),
        "percent_assigned": float(100 * assigned.mean()),
        "mean_probability": float(query_joint_result.obs[prob_col].mean()),
        "max_probability": float(query_joint_result.obs[prob_col].max()),
    })

joint_recovery_summary = pd.DataFrame(summary_rows)
display(joint_recovery_summary.round(4))
joint_recovery_summary.to_csv(
    ANALYSIS_DIR / "joint_scvi_AE_DE_recovery_summary.csv",
    index=False,
)

print("All transferred labels:")
print(query_joint_result.obs["predicted_leiden_sub_joint"].value_counts())


## 11. Copy annotations to the full-transcriptome object and save


In [ ]:
if not adata_full.obs_names.equals(query_joint_result.obs_names):
    raise ValueError("Full-gene and joint-query cell order differ.")

adata_full_joint = adata_full.copy()
adata_full_joint.obs["predicted_leiden_sub_joint"] = (
    query_joint_result.obs["predicted_leiden_sub_joint"].astype(str).to_numpy()
)
adata_full_joint.obs["prediction_confidence_joint"] = (
    query_joint_result.obs["prediction_confidence_joint"].to_numpy()
)
adata_full_joint.obsm["X_scVI_joint"] = query_joint_result.obsm["X_scVI_joint"].copy()
adata_full_joint.obsm["X_umap_joint"] = query_joint_result.obsm["X_umap_joint"].copy()

for col in query_joint_result.obs.columns:
    if col.startswith("prob_joint_"):
        adata_full_joint.obs[col] = query_joint_result.obs[col].to_numpy()

FINAL_FULL_JOINT = ANALYSIS_DIR / "amanda_liang_fullgenes_joint_scvi_annotated.h5ad"
FINAL_QUERY_JOINT = ANALYSIS_DIR / "amanda_liang_query_963genes_joint_scvi_annotated.h5ad"
FINAL_COMBINED_JOINT = ANALYSIS_DIR / "reference_plus_amanda_liang_joint_scvi_umap.h5ad"

# Convert object columns to strings before writing to avoid h5py serialization errors.
for obj in [adata_full_joint, query_joint_result, joint]:
    for col in obj.obs.select_dtypes(include=["object"]).columns:
        obj.obs[col] = obj.obs[col].astype(str)

adata_full_joint.write_h5ad(FINAL_FULL_JOINT)
query_joint_result.write_h5ad(FINAL_QUERY_JOINT)
joint.write_h5ad(FINAL_COMBINED_JOINT)

print("Saved full-gene joint annotation:", FINAL_FULL_JOINT)
print("Saved 963-gene joint query:", FINAL_QUERY_JOINT)
print("Saved combined joint UMAP:", FINAL_COMBINED_JOINT)


In [ ]:
# FIX OCM SAMPLES!!!

sample_correction = {
    "Chicago_Ctrl_72h": "Chicago_DOX_48h",
    "Chicago_DOX_48h": "Chicago_Ctrl_72h",
    "Michigan_Ctrl_72h": "Michigan_DOX_48h",
    "Michigan_DOX_48h": "Michigan_Ctrl_72h",
}

objects_to_correct = {
    "adata_full": adata_full,
    "adata_full_joint": adata_full_joint,
    "query": query,
    "query_joint": query_joint,
    "query_joint_result": query_joint_result,
}

for name, obj in objects_to_correct.items():
    if "sample_id" not in obj.obs.columns:
        print(f"{name}: no sample_id column")
        continue

    if "sample_id_original" not in obj.obs.columns:
        obj.obs["sample_id_original"] = obj.obs["sample_id"].astype(str)

    obj.obs["sample_id"] = (
        obj.obs["sample_id"]
        .astype(str)
        .replace(sample_correction)
    )

    print(f"\n{name}")
    print(obj.obs["sample_id"].value_counts())

In [ ]:
import numpy as np

for name, obj in objects_to_correct.items():
    if "sample_id" not in obj.obs.columns:
        continue

    obj.obs["dataset"] = np.where(
        obj.obs["sample_id"].str.contains("Chicago", case=False, na=False),
        "Chicago",
        "Michigan",
    )

    obj.obs["condition"] = np.where(
        obj.obs["sample_id"].str.contains("Ctrl", case=False, na=False),
        "Ctrl",
        "DOX",
    )

    obj.obs["timepoint"] = (
        obj.obs["sample_id"]
        .str.extract(r"(48h|72h)", expand=False)
    )

    print(f"\n{name}")
    display(
        obj.obs[
            [
                "sample_id_original",
                "sample_id",
                "dataset",
                "condition",
                "timepoint",
            ]
        ]
        .drop_duplicates()
        .sort_values("sample_id")
    )

In [ ]:
# Save corrected objects
adata_full.write_h5ad(
    "adata_full_OCM_labels_corrected.h5ad",
    compression="gzip",
)

adata_full_joint.write_h5ad(
    "adata_full_joint_OCM_labels_corrected.h5ad",
    compression="gzip",
)

query.write_h5ad(
    "query_OCM_labels_corrected.h5ad",
    compression="gzip",
)

query_joint_result.write_h5ad(
    "query_joint_result_OCM_labels_corrected.h5ad",
    compression="gzip",
)

## Interpretation

- This notebook does **not** remove cells based on chemistry.
- `joint_batch` allows the model to explain sample/run/chemistry-associated count differences as technical variation.
- Recovery of AE-LC or DE is evidence that the frozen-reference mapping was the problem, but marker co-expression and sample/time distributions must still be checked.
- If AE-LC remains absent despite strong coordinated AE markers, stop parameter tuning and ask the reference authors for the exact ribosomal-kit correction or final mapping pipeline.


In [ ]:
import pandas as pd

print("Probability array shape:", joint_probabilities.shape)

class_objects = {}

for name, obj in list(globals().items()):
    if hasattr(obj, "classes_"):
        print(name, obj.classes_)
        class_objects[name] = obj

In [ ]:
prob_df = pd.DataFrame(
    joint_probabilities,
    index=query_joint_result.obs_names,
    columns=knn_joint.classes_,
)

prob_df.head()

In [ ]:
labels = ["AE-LC", "DE", "PGC-LC", "CS-LC"]
predicted = query_joint_result.obs["predicted_leiden_sub_joint"]

rows = []

for label in labels:
    assigned = predicted.eq(label)
    probs = prob_df.loc[assigned, label]

    rows.append({
        "label": label,
        "n_assigned": int(assigned.sum()),
        "mean_assigned_probability": probs.mean(),
        "median_assigned_probability": probs.median(),
        "minimum_assigned_probability": probs.min(),
        "fraction_above_0.5": (probs >= 0.5).mean(),
        "fraction_above_0.8": (probs >= 0.8).mean(),
    })

confidence_summary = pd.DataFrame(rows)
confidence_summary

In [ ]:
for name, obj in list(globals().items()):
    if hasattr(obj, "n_obs") and hasattr(obj, "n_vars"):
        print(name, obj.shape)

In [ ]:
print(adata_full_joint.shape)

print(
    adata_full_joint.obs["predicted_leiden_sub_joint"]
    .value_counts(dropna=False)
)

In [ ]:
markers = [
    "POU5F1", "NANOG",
    "TBXT", "MIXL1", "EOMES",
    "MESP1", "PDGFRA", "HAND1",
    "TFAP2A", "TFAP2C", "GATA3", "ISL1",
    "SOX17", "FOXA2", "GATA6", "CER1",
    "NANOS3", "PRDM1",
]

present = [g for g in markers if g in adata_full_joint.var_names]
missing = [g for g in markers if g not in adata_full_joint.var_names]

print("Present:", present)
print("Missing:", missing)

sc.pl.dotplot(
    adata_full_joint,
    present,
    groupby="predicted_leiden_sub_joint",
    standard_scale="var",
    use_raw=False,
)

In [ ]:
ctrl = adata_full_joint[
    adata_full_joint.obs["sample_id"].str.contains("Ctrl")
].copy()

ctrl48 = adata_full_joint[
    adata_full_joint.obs["sample_id"].str.contains("Ctrl_48h")
].copy()

ctrl72 = adata_full_joint[
    adata_full_joint.obs["sample_id"].str.contains("Ctrl_72h")
].copy()

In [ ]:
import scanpy as sc

ctrl = adata_full_joint[
    adata_full_joint.obs["sample_id"].str.contains("Ctrl", case=False, na=False)
].copy()

sc.pl.dotplot(
    ctrl,
    ["APLNR", "APELA"],
    groupby="predicted_leiden_sub_joint",
    use_raw=False,
    expression_cutoff=0,
    mean_only_expressed=False,
    standard_scale="var",
    swap_axes=True,
    title="APLNR and APELA expression in control gastruloids",
)

In [ ]:
ctrl.obs["timepoint"] = (
    ctrl.obs["sample_id"]
    .str.extract(r"(48h|72h)", expand=False)
)

ctrl.obs["state_time"] = (
    ctrl.obs["predicted_leiden_sub_joint"].astype(str)
    + " | "
    + ctrl.obs["timepoint"].astype(str)
)

sc.pl.dotplot(
    ctrl,
    ["APLNR", "APELA"],
    groupby="state_time",
    use_raw=False,
    expression_cutoff=0,
    mean_only_expressed=False,
    standard_scale="var",
    swap_axes=True,
    title="APLNR and APELA by cell state and time",
)

In [ ]:
sc.pl.dotplot(
    ctrl,
    ["APLNR", "APELA"],
    groupby="state_time",
    use_raw=False,
    expression_cutoff=0,
    mean_only_expressed=True,
    swap_axes=True,
    title="APLNR and APELA detection by cell state and time",
)

In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse

adata = adata_full_joint

def get_gene_vector(adata, gene, layer="counts"):
    gene_idx = adata.var_names.get_loc(gene)

    if layer is None:
        x = adata.X[:, gene_idx]
    else:
        x = adata.layers[layer][:, gene_idx]

    if sparse.issparse(x):
        x = x.toarray()

    return np.asarray(x).ravel()

apl_counts = get_gene_vector(
    adata,
    "APLNR",
    layer="counts",
)

check = adata.obs[["sample_id"]].copy()
check["APLNR_count"] = apl_counts

apl_count_summary = (
    check.groupby("sample_id")["APLNR_count"]
    .agg(
        n_cells="size",
        n_positive=lambda x: int((x > 0).sum()),
        percent_positive=lambda x: 100 * (x > 0).mean(),
        total_counts="sum",
        mean_all="mean",
        mean_positive=lambda x: x[x > 0].mean() if (x > 0).any() else 0,
        median_positive=lambda x: x[x > 0].median() if (x > 0).any() else 0,
        max_count="max",
    )
)

display(apl_count_summary)

for sample in check["sample_id"].unique():
    x = check.loc[
        check["sample_id"] == sample,
        "APLNR_count",
    ]

    print("\n", sample)
    print(x.value_counts().sort_index().head(20))
    
    
check_genes = [
    "APLNR",
    "TBXT",
    "TBX6",
    "SOX2",
    "EPCAM",
    "RSPO3",
    "EVX1",
    "RIPPLY2",
    "APELA",
]

check_genes = [
    g for g in check_genes
    if g in adata.var_names
]

sc.pl.dotplot(
    adata,
    check_genes,
    groupby="sample_id",
    layer="counts",
    expression_cutoff=0,
    mean_only_expressed=True,
    swap_axes=True,
)